<div style="background-image: linear-gradient(rgba(10, 25, 47, 0.88), rgba(20, 45, 80, 0.88)), url('https://images.unsplash.com/photo-1518770660439-4636190af475?auto=format&fit=crop&w=1200&q=80'); background-size: cover; background-position: center; padding: 32px; border-radius: 12px; color: #ffffff; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.4); font-family: sans-serif;">
<h1 style="color: #ffffff; font-size: 2.2em; margin: 0 0 10px 0; font-weight: 700; text-align: center; text-shadow: 2px 2px 4px rgba(0,0,0,0.8);">GridWorld RL: Trajectory Profiling & Agent Policy Analytics</h1>
<p style="font-size: 1.05em; color: #64ffda; margin: 0 0 20px 0; text-align: center; text-shadow: 1px 1px 3px rgba(0,0,0,0.8); font-weight: 500;">Interactive State-Space Heatmaps, Episode Reward Dynamics & Multi-Class LightGBM Action Policy Predictor</p>
<hr style="border: none; border-top: 1px solid rgba(100, 255, 218, 0.3); margin: 15px 0;">
<h3 style="color: #e6f1ff; font-size: 1.25em; margin: 15px 0 10px 0; text-shadow: 1px 1px 3px rgba(0,0,0,0.8);">Table of Contents</h3>
<ol style="color: #ccd6f6; font-size: 1em; line-height: 1.8; margin: 0; padding-left: 20px; text-shadow: 1px 1px 3px rgba(0,0,0,0.8);">
<li><b>Section 1:</b> System Setup, Environment Ingestion & Deduplication</li>
<li><b>Section 2:</b> Interactive Spatial State Occupancy Heatmaps & Grid Analytics</li>
<li><b>Section 3:</b> Episode Trajectory Length & Cumulative Reward Dynamics</li>
<li><b>Section 4:</b> RL Domain Feature Engineering & Transition Ratios</li>
<li><b>Section 5:</b> Fast LightGBM Multi-Class Agent Action Policy Predictor</li>
<li><b>Section 6:</b> Policy Feature Importance & Agent Behaviour Takeaways</li>
</ol>

## 1: Environment Setup & Data Ingestion

In [1]:
import os
import glob
import warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

# Set rendering mode for public Kaggle viewing
pio.renderers.default = "iframe"

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Resolve dataset path dynamically
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True) or glob.glob('./*.csv')
if not csv_files:
    raise FileNotFoundError("No CSV files found in the working environment.")

target_file = max(csv_files, key=os.path.getsize)
print(f"Dataset Path Detected: {target_file}")

df = pd.read_csv(target_file)

# Standardize column headers and remove duplicate column labels
df.columns = [c.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for c in df.columns]
df = df.loc[:, ~df.columns.duplicated()].copy()

print(f"Schema Initialized Successfully. Active Trajectory Logs: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()

Dataset Path Detected: /kaggle/input/datasets/mobeenfatimah/gridworld-rl-trajectories100k-agent-decision-logs/synthetic_rl_trajectories.csv
Schema Initialized Successfully. Active Trajectory Logs: 100000 | Columns: 11


,episode_id,step,agent_type,state_json,action,action_name,reward,cumulative_reward,next_state_json,policy_entropy,done
0,EP-00001,1,PPO,"{""agent_pos"": [0, 0], ""target_pos"": [9, 9], ""d...",2,DOWN,1.0,1.0,"{""agent_pos"": [0, 1], ""target_pos"": [9, 9], ""d...",0.2213,False
1,EP-00001,2,PPO,"{""agent_pos"": [0, 1], ""target_pos"": [9, 9], ""d...",2,DOWN,1.0,2.0,"{""agent_pos"": [0, 2], ""target_pos"": [9, 9], ""d...",0.4315,False
2,EP-00001,3,PPO,"{""agent_pos"": [0, 2], ""target_pos"": [9, 9], ""d...",1,RIGHT,1.0,3.0,"{""agent_pos"": [1, 2], ""target_pos"": [9, 9], ""d...",0.2336,False
3,EP-00001,4,PPO,"{""agent_pos"": [1, 2], ""target_pos"": [9, 9], ""d...",2,DOWN,1.0,4.0,"{""agent_pos"": [1, 3], ""target_pos"": [9, 9], ""d...",0.1109,False
4,EP-00001,5,PPO,"{""agent_pos"": [1, 3], ""target_pos"": [9, 9], ""d...",1,RIGHT,1.0,5.0,"{""agent_pos"": [2, 3], ""target_pos"": [9, 9], ""d...",0.4420,False


## 2: Interactive Spatial Grid Occupancy & Trajectory Plots

In [2]:
# Capped sampling for fast Plotly rendering
MAX_SAMPLES = 10000
df_plot = df.sample(n=min(len(df), MAX_SAMPLES), random_state=42) if len(df) > MAX_SAMPLES else df

numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_plot.select_dtypes(include=['object', 'category']).columns.tolist()

# Identify spatial coordinates and RL decision attributes dynamically
x_col = next((c for c in numeric_cols if 'x' in c or 'col' in c or 'pos_x' in c), numeric_cols[0] if numeric_cols else None)
y_col = next((c for c in numeric_cols if 'y' in c or 'row' in c or 'pos_y' in c and c != x_col), numeric_cols[1] if len(numeric_cols) > 1 else None)
action_col = next((c for c in df_plot.columns if 'action' in c or 'move' in c or 'step' in c), categorical_cols[0] if categorical_cols else None)
reward_col = next((c for c in numeric_cols if 'reward' in c or 'score' in c or 'penalty' in c), numeric_cols[-1] if numeric_cols else None)

# 1. Spatial Occupancy Heatmap in 2D Grid Space
if x_col and y_col:
    grid_counts = df_plot.groupby([x_col, y_col]).size().reset_index(name='occupancy_count')
    pivot_grid = grid_counts.pivot(index=y_col, columns=x_col, values='occupancy_count').fillna(0)
    
    fig_heatmap = px.imshow(
        pivot_grid,
        labels=dict(x="Grid X Coordinate", y="Grid Y Coordinate", color="Visit Frequency"),
        title='<b>Section 2: Interactive GridWorld State Occupancy Density Heatmap</b>',
        template='plotly_dark',
        color_continuous_scale='Magma',
        height=500
    )
    fig_heatmap.show()

In [3]:

# 2. Action Distribution by Spatial Coordinate
if x_col and action_col:
    fig_scatter = px.scatter(
        df_plot.head(1000),
        x=x_col,
        y=y_col if y_col else reward_col,
        color=action_col,
        title=f'<b>Agent Action Choice Map in Grid Space</b>',
        template='plotly_dark',
        height=450
    )
    fig_scatter.update_traces(marker=dict(size=6, opacity=0.8))
    fig_scatter.show()

## 3: Trajectory Length & Reward Convergence Profiling

In [4]:
episode_col = next((c for c in df_plot.columns if 'episode' in c or 'traj' in c or 'run' in c), None)

# 1. Cumulative Reward Distribution Across Episodes
if reward_col:
    fig_box = px.box(
        df_plot,
        x=action_col if action_col else None,
        y=reward_col,
        color=action_col if action_col else None,
        points='outliers',
        title='<b>Section 3: Reward Variance Across Agent Actions</b>',
        template='plotly_dark',
        height=420
    )
    fig_box.update_layout(showlegend=False)
    fig_box.show()

In [5]:

# 2. Episode Duration or Trajectory Step Counts
if episode_col:
    ep_summary = df_plot.groupby(episode_col).size().reset_index(name='trajectory_length').head(50)
    fig_bar = px.bar(
        ep_summary,
        x=episode_col,
        y='trajectory_length',
        color='trajectory_length',
        color_continuous_scale='Viridis',
        title='<b>Trajectory Step Count per Episode</b>',
        template='plotly_dark',
        height=400
    )
    fig_bar.show()

## 4: RL Trajectory Feature Engineering

In [6]:
df_engineered = df.copy()

# 1. State Delta Feature Engineering (Distance from Origin)
if x_col in df_engineered.columns and y_col in df_engineered.columns:
    df_engineered['manhattan_dist_from_origin'] = np.abs(df_engineered[x_col]) + np.abs(df_engineered[y_col])
    df_engineered['euclidean_dist_from_origin'] = np.sqrt(df_engineered[x_col]**2 + df_engineered[y_col]**2)

# 2. Step Reward Efficiency Metrics
step_col = next((c for c in numeric_cols if 'step' in c or 't' in c or 'index' in c), None)
if reward_col in df_engineered.columns and step_col in df_engineered.columns:
    df_engineered['reward_per_step'] = df_engineered[reward_col] / (df_engineered[step_col] + 1)

print("RL Trajectory Feature Engineering Complete.")
df_engineered.head(3)

RL Trajectory Feature Engineering Complete.


,episode_id,step,agent_type,state_json,action,action_name,reward,cumulative_reward,next_state_json,policy_entropy,done,manhattan_dist_from_origin,euclidean_dist_from_origin,reward_per_step
0,EP-00001,1,PPO,"{""agent_pos"": [0, 0], ""target_pos"": [9, 9], ""d...",2,DOWN,1.0,1.0,"{""agent_pos"": [0, 1], ""target_pos"": [9, 9], ""d...",0.2213,False,1.2213,1.024194,0.500000
1,EP-00001,2,PPO,"{""agent_pos"": [0, 1], ""target_pos"": [9, 9], ""d...",2,DOWN,1.0,2.0,"{""agent_pos"": [0, 2], ""target_pos"": [9, 9], ""d...",0.4315,False,2.4315,2.046019,0.333333
2,EP-00001,3,PPO,"{""agent_pos"": [0, 2], ""target_pos"": [9, 9], ""d...",1,RIGHT,1.0,3.0,"{""agent_pos"": [1, 2], ""target_pos"": [9, 9], ""d...",0.2336,False,3.2336,3.009081,0.250000


## 5: Fast LightGBM Agent Action Policy Predictor

In [7]:
# Identify policy target variable
target_col = action_col if action_col else df_engineered.columns[-1]

# Capped sample for sub-second classification execution
MAX_MODEL_SAMPLES = 15000
df_ml = df_engineered.sample(n=min(len(df_engineered), MAX_MODEL_SAMPLES), random_state=42) if len(df_engineered) > MAX_MODEL_SAMPLES else df_engineered

X = df_ml.drop(columns=[target_col], errors='ignore')
y = df_ml[target_col]

# Encode target categorical action labels to integers
if y.dtype == 'object' or str(y.dtype) == 'category':
    y, target_labels = pd.factorize(y)

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Dense Column Transformer with fast OneHotEncoding
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=None)

# Fast Multi-Class LightGBM Action Predictor
lgb_cls = Pipeline([
    ('preproc', preprocessor), 
    ('classifier', lgb.LGBMClassifier(
        n_estimators=45,
        learning_rate=0.09, 
        num_leaves=18,
        max_depth=5,
        n_jobs=-1,
        random_state=42, 
        verbose=-1
    ))
])

lgb_cls.fit(X_train, y_train)

y_pred = lgb_cls.predict(X_test)

results = pd.DataFrame([{
    'Model': 'LightGBM RL Action Policy Predictor (Fast)', 
    'Accuracy': round(accuracy_score(y_test, y_pred), 4)
}])

print("Section 5: RL Agent Action Prediction Model Results")
results

Section 5: RL Agent Action Prediction Model Results


,Model,Accuracy
0,LightGBM RL Action Policy Predictor (Fast),0.19


## 6: Feature Importance Analysis

In [8]:
lgb_model = lgb_cls.named_steps['classifier']
preproc_step = lgb_cls.named_steps['preproc']

ohe_cols = preproc_step.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_features).tolist() if cat_features else []
all_features = num_features + ohe_cols

imp_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': lgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

fig_imp = px.bar(
    imp_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='<b>Section 6: Top State Drivers of Agent Action Decisions</b>',
    color='Importance',
    color_continuous_scale='Viridis',
    template='plotly_dark',
    height=420
)
fig_imp.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_imp.show()

<div style="background-image: linear-gradient(rgba(10, 25, 47, 0.90), rgba(20, 45, 80, 0.90)), url('https://images.unsplash.com/photo-1518770660439-4636190af475?auto=format&fit=crop&w=1200&q=80'); background-size: cover; background-position: center; padding: 32px; border-radius: 12px; color: #ffffff; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.4); font-family: sans-serif;">
<h2 style="color: #64ffda; font-size: 1.8em; margin: 0 0 10px 0; font-weight: 700; text-align: center; text-shadow: 2px 2px 4px rgba(0,0,0,0.8);">Conclusion & Policy Analytics Takeaways</h2>
<p style="font-size: 1.05em; color: #8892b0; margin: 0 0 20px 0; text-align: center; text-shadow: 1px 1px 3px rgba(0,0,0,0.8);">Key Insights from Reinforcement Learning Trajectory Logs</p>
<hr style="border: none; border-top: 1px solid rgba(100, 255, 218, 0.3); margin: 18px 0;">
<ul style="color: #ccd6f6; font-size: 1em; line-height: 1.8; margin: 0; padding-left: 20px; text-shadow: 1px 1px 3px rgba(0,0,0,0.8);">
<li><b>State Occupancy Bottlenecks:</b> Density heatmaps uncover specific spatial coordinate clusters where the RL agent spends disproportionate decision steps.</li>
<li><b>Distance & Reward Signals:</b> Engineered state metrics (Euclidean distance, step efficiency) provide strong predictive power for imitating the agent's learned policy.</li>
<li><b>High-Speed Action Imitation:</b> LightGBM multi-class boosting accurately predicts agent action selections across large trajectory decision logs in sub-second inference runs.</li>
</ul>